In [ ]:
import time
import json
import os
import contextlib
import pandas as pd
from typing import TypedDict
from enum import Enum

import wave
import torch
import whisper.timing as whisper_timing
from batchalign.pipelines.pipeline import BatchalignPipeline
from batchalign import Document
from batchalign.formats.chat import CHATFile

In [ ]:
class TorchBackend(Enum):
    MPS = "mps"  # Apple Silicon (M1/M2/M3/M4/M5 Macbooks)
    CUDA = "cuda"  # Nvidia (dedicated windows GPU)
    CPU = "cpu"  # CPU (fallback, slower)


def get_device_and_dtype() -> tuple[TorchBackend, torch.dtype]:
    if torch.backends.mps.is_available():
        # Macbook with Apple Silicon
        device = TorchBackend.MPS
        torch_dtype = torch.float32
    elif torch.cuda.is_available():
        # Windows with Nvidia GPU
        device = TorchBackend.CUDA
        torch_dtype = torch.float16
    else:
        # CPU fallback
        device = TorchBackend.CPU
        torch_dtype = torch.float32
    return device, torch_dtype


def build_pipeline(lang="fra", num_speakers=2):
    device, torch_dtype = get_device_and_dtype()

    # print("MPS disponible :", torch.backends.mps.is_available())
    # print("CUDA disponible :", torch.cuda.is_available())
    # print("Device détecté :", device.value)
    # print("Type détecté :", torch_dtype)
    # print("Création du pipeline Batchalign...")

    # API compatible avec la doc actuelle
    nlp = BatchalignPipeline.new("asr", lang=lang, num_speakers=num_speakers)  # type: ignore
    # engine specs: [('asr', 'whisper_oai'), ('disfluency', 'replacement'), ('retracing', 'ngram')]
    # -> whisper-large-v3 whisper.load_model("turbo")

    return nlp, device, torch_dtype


def get_wav_duration_seconds(wav_path: str) -> float:
    """Retourne la durée d'un fichier WAV en secondes."""
    with contextlib.closing(wave.open(wav_path, "r")) as f:
        frames = f.getnframes()
        rate = f.getframerate()
        if rate > 0:
            return frames / float(rate)
    return -1


build_pipeline()

In [ ]:
class TranscriptionResult(TypedDict):
    audio_file: str
    torch_backend: str
    torch_dtype: str
    audio_duration_s: float
    pipeline_creation_time_s: float
    transcription_time_s: float


def force_using_mps(nlp: BatchalignPipeline):
    """Force le modèle Whisper à utiliser le backend MPS sur Macbooks Apple Silicon."""

    # Set Torch MPS fallback to allow using MPS even if some operations are not supported
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

    # Convert the Whisper model to MPS
    model = nlp.__dict__["_BatchalignPipeline__generator"].__dict__[
        "_OAIWhisperEngine__whisper"
    ]
    model = model.to("mps")
    nlp.__dict__["_BatchalignPipeline__generator"].__dict__[
        "_OAIWhisperEngine__whisper"
    ] = model
    p = next(model.parameters())
    # print("device:", p.device)
    # print("dtype:", p.dtype)

    # Patch whisper DTW to use CPU because of unsupported operations on MPS
    _original_dtw = whisper_timing.dtw

    def dtw_mps_safe(x):
        if isinstance(x, torch.Tensor) and x.device.type == "mps":
            # print("Using CPU fallback for DTW on MPS device")
            return whisper_timing.dtw_cpu(x.detach().cpu().double().numpy())

        return _original_dtw(x)

    whisper_timing.dtw = dtw_mps_safe


def transcribe_audio(
    audio_file: str, torch_backend: TorchBackend, outfile: str
) -> TranscriptionResult:

    # Build the pipeline
    t0 = time.time()
    nlp, device, torch_dtype = build_pipeline()
    t_pipeline = time.time() - t0

    # Extract audio duration
    audio_duration = get_wav_duration_seconds(audio_file)

    # Force MPS usage if on Apple Silicon
    if torch_backend == TorchBackend.MPS:
        force_using_mps(nlp)
    elif torch_backend == TorchBackend.CUDA:
        raise NotImplementedError("CUDA backend is not yet implemented.")

    # Create a Batchalign Document from the audio file
    doc = Document.new(media_path=audio_file, lang="fra")

    t0 = time.time()
    doc = nlp(doc)
    t_transcription = time.time() - t0

    chat = CHATFile(doc=doc)
    chat.write(outfile)

    print(f">>> Transcription of {audio_file} <<<")
    print(f"\t> Whisper backend:  \t{torch_backend.value} ({torch_dtype})")
    print(f"\t> Audio duration:   \t{audio_duration:.2f} s")
    print(f"\t> Pipeline creation:\t{t_pipeline:.2f} s")
    print(f"\t> Transcription time:\t{t_transcription:.2f} s")
    return TranscriptionResult(
        audio_file=audio_file,
        torch_backend=torch_backend.value,
        torch_dtype=str(torch_dtype),
        audio_duration_s=audio_duration,
        pipeline_creation_time_s=t_pipeline,
        transcription_time_s=t_transcription,
    )


transcribe_audio(
    "./data/sample/begaiement.wav",
    TorchBackend.MPS,
    "./data/sample/output.cha",
)

In [ ]:
def process_folder(
    input_folder: str, output_folder: str, overwrite_existing: bool = False
):

    for file in os.listdir(input_folder):
        if not file.endswith(".wav"):
            continue

        participant_id, task_version, task_name = file.split("_")
        outfile = (
            f"{output_folder}/output/{participant_id}_{task_version}_{task_name}.cha"
        )

        for backend in [TorchBackend.MPS, TorchBackend.CPU]:

            metadata_file = f"{output_folder}/metadata/{backend.value}_{participant_id}_{task_version}_{task_name}.json"

            if (
                os.path.isfile(metadata_file)
                and os.path.isfile(outfile)
                and not overwrite_existing
            ):
                print(
                    f"> Skipping {file} with backend {backend.value} (already exists)"
                )
                continue

            try:
                result = transcribe_audio(
                    f"{input_folder}/{file}",
                    backend,
                    outfile,
                )

                with open(metadata_file, "w") as f:
                    json.dump(result, f, indent=4)
                print(f"> Transcription metadata saved to {metadata_file}")

            except Exception as e:
                print(f"Error processing {file}: {e}")
                continue


process_folder("./data/input", "./data/batchalign", overwrite_existing=False)

In [ ]:
def aggregate_results(metadata_folder: str) -> pd.DataFrame:
    data = []
    for file in os.listdir(metadata_folder):

        with open(f"{metadata_folder}/{file}", "r") as f:
            d = json.load(f)
        data += [d]

    df = pd.DataFrame(data)
    df["audio_file"] = df["audio_file"].str.lstrip("./data/input/")
    df["real_time_factor"] = 1 / (df["transcription_time_s"] / df["audio_duration_s"])
    del df["torch_dtype"]
    del df["pipeline_creation_time_s"]
    return df.sort_values(["audio_file", "torch_backend"])


df = aggregate_results("./data/batchalign/metadata")
df.to_csv("./data/batchalign/batchalign_results.csv", index=False)
df